<a href="https://colab.research.google.com/github/arjunverma2004/LLM-Finetuning/blob/main/LiquidAI_finetune_prac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Self code Llama fine tune

In [1]:
from datasets import load_dataset

dataset = load_dataset("Annanay/aml_song_lyrics_balanced", split="train")

README.md:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

training.csv:   0%|          | 0.00/29.6M [00:00<?, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/12196 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2539 [00:00<?, ? examples/s]

In [2]:
!pip install -U bitsandbytes accelerate peft transformers trl


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.9/532.9 kB 11.6 MB/s eta 0:00:00


In [3]:
model_name = "Qwen/Qwen3-0.6B"

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type = "nf4",
    bnb_4bit_compute_dtype = torch.float16,
    bnb_4bit_use_double_quant = True,
)

model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config = bnb_config,   # THIS is quantization
    device_map = "auto",
    trust_remote_code = True,
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

### Data Processing




In [ ]:
### For example only ##
from transformers import AutoTokenizer
from datasets import load_dataset


# Load a tokenizer to use its chat template
template_tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")

def format_prompt(example):
    """Format the prompt to using the <|user|> template TinyLLama is using"""

    # Format answers
    chat = example["messages"]
    prompt = template_tokenizer.apply_chat_template(chat, tokenize=False)

    return {"text": prompt}

# Load and format the data using the template TinyLLama is using
dataset = (
    load_dataset("HuggingFaceH4/ultrachat_200k",  split="test_sft")
      .shuffle(seed=42)
      .select(range(3_000))
)
dataset = dataset.map(format_prompt)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Generating train_sft split:   0%|          | 0/207865 [00:00<?, ? examples/s]

Generating test_sft split:   0%|          | 0/23110 [00:00<?, ? examples/s]

Generating train_gen split:   0%|          | 0/256032 [00:00<?, ? examples/s]

Generating test_gen split:   0%|          | 0/28304 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [5]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)


In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r = 8,
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM",
    target_modules = [
        "q_proj", "k_proj", "v_proj",
        "o_proj", "up_proj", "down_proj", "gate_proj"
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 5,046,272 || all params: 601,096,192 || trainable%: 0.8395


sft_trainer.jpeg

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,   # raw text
    processing_class = tokenizer,
    formatting_func = lambda x: x["lyrics"],
    args = TrainingArguments(
        output_dir = "qwen3_lyrics_4bit",
        per_device_train_batch_size = 2,   # 👈 T4-safe
        gradient_accumulation_steps = 8,
        learning_rate = 2e-4,
        max_steps = 150,
        bf16 = True,
        fp16 = False,
        logging_steps = 10,
        save_steps =25,
        optim = "paged_adamw_8bit",  # 🔥 memory saver
        report_to = "none",
    ),
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/transformers/training_args.py:2111: FutureWarning: `--push_to_hub_token` is deprecated and will be removed in version 5 of 🤗 Transformers. Use `--hub_token` instead.
  warnings.warn(


Tokenizing train dataset:   0%|          | 0/12196 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12196 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,2.140400
20,2.206900
30,2.091500
40,2.157300
50,2.219500
60,2.096300
70,2.161100
80,2.035200
90,2.078200
100,2.092400


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1044: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=150, training_loss=2.1517694091796873, metrics={'train_runtime': 2568.7603, 'train_samples_per_second': 0.934, 'train_steps_per_second': 0.058, 'total_flos': 2931419740372992.0, 'train_loss': 2.1517694091796873})

In [9]:
model.save_pretrained("qwen3_lyrics_finetuned")
tokenizer.save_pretrained("qwen3_lyrics_finetuned")


('qwen3_lyrics_finetuned/tokenizer_config.json',
 'qwen3_lyrics_finetuned/special_tokens_map.json',
 'qwen3_lyrics_finetuned/chat_template.jinja',
 'qwen3_lyrics_finetuned/vocab.json',
 'qwen3_lyrics_finetuned/merges.txt',
 'qwen3_lyrics_finetuned/added_tokens.json',
 'qwen3_lyrics_finetuned/tokenizer.json')

### Merge Adapter

In [10]:
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    "qwen3_lyrics_finetuned",
    low_cpu_mem_usage=True,
    device_map="auto",
)

# Merge LoRA and base model
merged_model = model.merge_and_unload()

In [63]:
from transformers import pipeline

# Use our predefined prompt template
prompt = """You are an artist and songwriter. I will give you the opening lines of a song.
    Your task is to complete the song with full lyrics, including verses, choruses,
    and possibly a bridge and outro, ensuring a coherent and complete narrative.\n\n
    Complete the song lyrics:\n
    Start:\n"
    If I was you, I would cut up my wrist (dumb bit')\n
    XO tatted all over her body, oh (yeah)\n
    She just wanna roll and I don't mind it, yeah
"""
dic1 = [{'role': "user", 'content': prompt}]

prompt2 = tokenizer.apply_chat_template(dic1, tokenize=False, add_generation_prompt=True)

# Run our instruction-tuned model with sampling parameters
pipe = pipeline(
    task="text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    max_new_tokens=2048,
    do_sample=True,
    temperature=0.7,
    top_p=0.9
)
print(pipe(prompt2)[0]["generated_text"])

Device set to use cuda:0


<|im_start|>user
You are an artist and songwriter. I will give you the opening lines of a song. 
    Your task is to complete the song with full lyrics, including verses, choruses, 
    and possibly a bridge and outro, ensuring a coherent and complete narrative.


    Complete the song lyrics:

    Start:
"
    If I was you, I would cut up my wrist (dumb bit')

    XO tatted all over her body, oh (yeah)

    She just wanna roll and I don't mind it, yeah
<|im_end|>
<|im_start|>assistant
<think>
Okay, let's start by looking at the initial lines of the song. The first line is "If I was you, I would cut up my wrist (dumb bit')". The user wants me to complete the song with full lyrics including verses, choruses, a bridge, and an outro. 

First, I need to structure the song properly. The user mentioned including verses, choruses, a bridge, and an outro. So I should break the song into these sections. Let me start by looking at the first part of the song. The user's first line is "If I was yo

In [60]:
ltest = dataset['lyrics'][10]
ltest
dic1 = [{'role': "user", 'content': ltest}]
dic1

[{'role': 'user',
  'content': "'I Want To Be Alone Dialogue LyricsI want to be alone\\nI need to touch each stone\\nFace the grave that I have grown\\nI want to be\\nAlone\\nBefore all the days are gone\\nAnd darker walls are bent and torn\\nTo pass the time of those who mourn\\nI want to be\\nAlone\\nRivers that run anywhere\\nAre in my hand and just up the stair\\nPast the eyes of those who care\\nWho can never be\\nAlone\\nChanges that were not meant to be\\nTow the hours of my memory\\nSing a song of love to me\\nTo say you must never\\nNever be alone\\nThe tears of a silent rain\\nSeek shelter on my broken pain\\nAnd run away\\nBut I remain\\nTo speak the words\\nThat sing\\nOf alone\\nI want to be alone\\nI need to touch each stone\\nFace the grave that I have grown\\nI want to be\\nAloneYou might also like6Embed', 'default'"}]

In [61]:
test = tokenizer.apply_chat_template(dic1, tokenize=False, add_generation_prompt=True)

In [62]:
test

"<|im_start|>user\n'I Want To Be Alone Dialogue LyricsI want to be alone\\nI need to touch each stone\\nFace the grave that I have grown\\nI want to be\\nAlone\\nBefore all the days are gone\\nAnd darker walls are bent and torn\\nTo pass the time of those who mourn\\nI want to be\\nAlone\\nRivers that run anywhere\\nAre in my hand and just up the stair\\nPast the eyes of those who care\\nWho can never be\\nAlone\\nChanges that were not meant to be\\nTow the hours of my memory\\nSing a song of love to me\\nTo say you must never\\nNever be alone\\nThe tears of a silent rain\\nSeek shelter on my broken pain\\nAnd run away\\nBut I remain\\nTo speak the words\\nThat sing\\nOf alone\\nI want to be alone\\nI need to touch each stone\\nFace the grave that I have grown\\nI want to be\\nAloneYou might also like6Embed', 'default'<|im_end|>\n<|im_start|>assistant\n"